# DeepTruth — Stage 5-6: Adversarial Hardening + Confidence Calibration

**Run on Google Colab Pro (A100 GPU recommended)**

### Stages
5. **Adversarial Training**: Fine-tune with FGSM/PGD adversarial examples to resist attacks
6. **Temperature Scaling**: Calibrate confidence scores so `P(fake)=0.9` actually means 90% reliable

### Prerequisites
- Complete notebooks 02, 03, 04
- Both `deeptruth_image_model_final.pth` and `deeptruth_video_model_final.pth` in Drive/checkpoints

In [ ]:
# Cell 1 — Setup
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm open-clip-torch albumentations einops

import os, json, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from torchmetrics import AUROC, CalibrationError

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DRIVE_ROOT = '/content/drive/MyDrive/DeepTruth'
DATA_ZIP   = f'{DRIVE_ROOT}/deeptruth_data.zip'
DATA_DIR   = '/content/deeptruth_data'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2 — Unpack data (if not already done)
if not os.path.exists(DATA_DIR):
    !unzip -q {DATA_ZIP} -d {DATA_DIR}
    print('Unpacked.')

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

FAKE_TYPE_MAP = {
    'real': 0, 'Deepfakes': 1, 'Face2Face': 2, 'FaceSwap': 3,
    'NeuralTextures': 4, 'DeepFakeDetection': 5, 'FaceShifter': 6,
    'gan_generated': 7, 'diffusion_generated': 8, 'unknown_fake': 9,
}

def get_val_transform():
    return A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

# Pixel-space bounds for PGD (after normalisation)
MEAN_T = torch.tensor(IMAGENET_MEAN).view(3, 1, 1).to(DEVICE)
STD_T  = torch.tensor(IMAGENET_STD).view(3,  1, 1).to(DEVICE)

def denorm(x):
    return x * STD_T + MEAN_T

def renorm(x):
    return (x - MEAN_T) / STD_T

print('Setup complete.')

In [ ]:
# Cell 3 — Minimal dataset for adversarial fine-tuning
import sys

class SimpleFaceDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.transform = transform
        self.samples = []
        for label_name, label in [('real', 0), ('fake', 1)]:
            d = os.path.join(root_dir, label_name)
            if not os.path.isdir(d):
                continue
            files = sorted([f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.png'))])
            for f in files:
                self.samples.append((os.path.join(d, f), label))
        random.seed(42)
        random.shuffle(self.samples)
        n = len(self.samples)
        if split == 'val':
            self.samples = self.samples[int(0.8*n):int(0.9*n)]
        elif split == 'test':
            self.samples = self.samples[int(0.9*n):]
        elif split == 'adv_train':
            # Use a subset for adversarial fine-tuning (save time)
            self.samples = self.samples[:min(20000, int(0.8*n))]
        else:
            self.samples = self.samples[:int(0.8*n)]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        img_np = np.array(img)
        if self.transform:
            img_np = self.transform(image=img_np)['image'].float()
        return img_np, torch.tensor(label, dtype=torch.long)


# Find face crops directory
FACE_DIR = None
for candidate in [
    os.path.join(DATA_DIR, 'face_crops'),
    os.path.join(DATA_DIR, 'raw'),
    DATA_DIR,
]:
    if os.path.exists(os.path.join(candidate, 'real')) and \
       os.path.exists(os.path.join(candidate, 'fake')):
        FACE_DIR = candidate
        break

print(f'Face dir: {FACE_DIR}')
adv_ds  = SimpleFaceDataset(FACE_DIR, split='adv_train', transform=get_val_transform())
val_ds  = SimpleFaceDataset(FACE_DIR, split='val',       transform=get_val_transform())
test_ds = SimpleFaceDataset(FACE_DIR, split='test',      transform=get_val_transform())

adv_loader  = DataLoader(adv_ds,  batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader  = DataLoader(val_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f'Adv train: {len(adv_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# Cell 4 — Load model (run 02_build_model cells first)

IMAGE_CKPT = f'{CKPT_DIR}/deeptruth_image_model_final.pth'

model = DeepTruthHybridV2(num_fake_types=len(FAKE_TYPE_MAP), dropout=0.3).to(DEVICE)
ckpt  = torch.load(IMAGE_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'], strict=False)
print(f'Loaded image model. Test AUROC was: {ckpt.get("test_metrics", {}).get("auroc", "N/A")}')

In [ ]:
# Cell 5 — Adversarial attack implementations

def fgsm_attack(model, imgs, labels, eps=4/255):
    """Fast Gradient Sign Method — single-step attack."""
    imgs_adv = imgs.clone().detach().requires_grad_(True)
    out = model(imgs_adv)
    loss = F.cross_entropy(out['binary_logit'], labels)
    loss.backward()
    perturbation = eps * imgs_adv.grad.sign()
    # Apply in pixel space to keep valid range
    imgs_pixel = denorm(imgs) + perturbation
    imgs_pixel = imgs_pixel.clamp(0, 1)
    return renorm(imgs_pixel).detach()


def pgd_attack(model, imgs, labels, eps=4/255, alpha=1/255, steps=10):
    """Projected Gradient Descent — strongest white-box attack."""
    imgs_adv = imgs.clone().detach()
    imgs_adv = imgs_adv + torch.empty_like(imgs_adv).uniform_(-eps, eps)  # random init

    for _ in range(steps):
        imgs_adv.requires_grad_(True)
        out = model(imgs_adv)
        loss = F.cross_entropy(out['binary_logit'], labels)
        model.zero_grad()
        loss.backward()
        grad_sign = imgs_adv.grad.sign()
        imgs_adv = imgs_adv.detach() + alpha * grad_sign

        # Project back into epsilon ball (pixel space)
        delta = renorm(denorm(imgs_adv).clamp(0,1)) - imgs
        delta_pixel = denorm(imgs_adv) - denorm(imgs)
        delta_pixel = delta_pixel.clamp(-eps, eps)
        imgs_pixel  = (denorm(imgs) + delta_pixel).clamp(0, 1)
        imgs_adv    = renorm(imgs_pixel).detach()

    return imgs_adv


def test_robustness(model, loader, attack_fn, device, n_batches=20):
    """Measure accuracy under adversarial attack."""
    model.eval()
    correct, total = 0, 0
    for i, (imgs, labels) in enumerate(loader):
        if i >= n_batches: break
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.enable_grad():
            imgs_adv = attack_fn(model, imgs, labels)
        with torch.no_grad():
            out = model(imgs_adv)
            preds = out['binary_logit'].argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total


# Baseline robustness before adversarial training
print('Evaluating baseline robustness...')
fgsm_acc = test_robustness(model, val_loader,
                           lambda m,x,y: fgsm_attack(m,x,y,eps=4/255),
                           DEVICE)
print(f'FGSM (eps=4/255) accuracy: {fgsm_acc:.4f}')

In [ ]:
# Cell 6 — Stage 5: Adversarial fine-tuning (TRADES-style)
# TRADES: combine clean loss + KL divergence between clean and adversarial predictions

ADV_EPOCHS   = 5
ADV_LR       = 5e-6
ADV_BETA     = 6.0   # trade-off: higher = more robustness, lower = more accuracy
EPS          = 4/255
ADV_CKPT     = f'{CKPT_DIR}/stage5_adversarial_best.pth'

# Only fine-tune heads and fusion (backbones stay frozen for speed)
for name, param in model.named_parameters():
    if any(k in name for k in ['clip_stream', 'effnet_stream']):
        param.requires_grad = False
    else:
        param.requires_grad = True

adv_params = [p for p in model.parameters() if p.requires_grad]
opt_adv = torch.optim.AdamW(adv_params, lr=ADV_LR, weight_decay=1e-4)
scaler_adv = GradScaler()

best_adv_auroc = 0
history_adv    = []

print(f'Adversarial fine-tuning: {ADV_EPOCHS} epochs, eps={EPS:.4f}, beta={ADV_BETA}')

for epoch in range(ADV_EPOCHS):
    model.train()
    running = {'clean': 0, 'adv': 0, 'total': 0}
    pbar = tqdm(adv_loader, desc=f'Adv E{epoch+1}/{ADV_EPOCHS}')

    for imgs, labels in pbar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        # Generate adversarial examples (PGD-5 for speed)
        model.eval()  # Must eval for BN consistency during attack
        with torch.enable_grad():
            imgs_adv = pgd_attack(model, imgs, labels, eps=EPS, alpha=EPS/4, steps=5)
        model.train()

        opt_adv.zero_grad()
        with autocast():
            # Clean forward
            out_clean = model(imgs)
            loss_clean = F.cross_entropy(out_clean['binary_logit'], labels)

            # Adversarial forward
            out_adv = model(imgs_adv)
            # KL divergence between clean and adversarial predictions (TRADES)
            p_clean = F.softmax(out_clean['binary_logit'].detach(), dim=1)
            p_adv   = F.log_softmax(out_adv['binary_logit'], dim=1)
            loss_adv = F.kl_div(p_adv, p_clean, reduction='batchmean')

            loss = loss_clean + ADV_BETA * loss_adv

        scaler_adv.scale(loss).backward()
        scaler_adv.unscale_(opt_adv)
        torch.nn.utils.clip_grad_norm_(adv_params, 1.0)
        scaler_adv.step(opt_adv)
        scaler_adv.update()

        running['clean'] += loss_clean.item()
        running['adv']   += loss_adv.item()
        running['total'] += loss.item()
        pbar.set_postfix(clean=f'{loss_clean.item():.3f}', adv=f'{loss_adv.item():.3f}')

    # Evaluate clean + robust accuracy
    model.eval()
    auroc_m = AUROC(task='binary').to(DEVICE)
    all_probs, all_labels_list = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            probs = torch.softmax(out['binary_logit'], dim=1)[:, 1]
            all_probs.append(probs)
            all_labels_list.append(labels)
    auroc = auroc_m(torch.cat(all_probs), torch.cat(all_labels_list)).item()

    fgsm_rob = test_robustness(model, val_loader,
                               lambda m,x,y: fgsm_attack(m,x,y,eps=EPS),
                               DEVICE, n_batches=10)

    n = len(adv_loader)
    history_adv.append({'epoch': epoch+1, 'auroc': auroc, 'fgsm_rob': fgsm_rob,
                        'loss_clean': running['clean']/n, 'loss_adv': running['adv']/n})
    print(f'  Adv E{epoch+1}: clean_auroc={auroc:.4f} fgsm_rob={fgsm_rob:.4f}')

    if auroc > best_adv_auroc:
        best_adv_auroc = auroc
        torch.save({'model_state': model.state_dict(), 'auroc': auroc}, ADV_CKPT)
        print(f'    -> Best saved.')

print(f'Adversarial training complete. Best AUROC: {best_adv_auroc:.4f}')

In [ ]:
# Cell 7 — Robustness comparison: before vs after adversarial training

ckpt_adv = torch.load(ADV_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt_adv['model_state'])

print('=== Robustness Report ===')
for eps_val in [2/255, 4/255, 8/255]:
    fgsm_acc_after = test_robustness(model, val_loader,
                                     lambda m,x,y: fgsm_attack(m,x,y,eps=eps_val),
                                     DEVICE, n_batches=15)
    pgd_acc_after  = test_robustness(model, val_loader,
                                     lambda m,x,y: pgd_attack(m,x,y,eps=eps_val,steps=20),
                                     DEVICE, n_batches=10)
    print(f'  eps={eps_val*255:.0f}/255 — FGSM: {fgsm_acc_after:.4f} | PGD-20: {pgd_acc_after:.4f}')

In [ ]:
# Cell 8 — Stage 6: Temperature Scaling (confidence calibration)
# A single scalar T scales logits: p = softmax(logits / T)
# T > 1 -> less confident | T < 1 -> more confident
# Optimal T found by minimizing NLL on validation set (no model weights change)

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    def calibrate(self, model, val_loader, device, max_iter=100):
        """Find optimal temperature on validation set."""
        self.to(device)
        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=max_iter)

        # Collect all logits from validation set
        all_logits, all_labels = [], []
        model.eval()
        with torch.no_grad():
            for imgs, labels in tqdm(val_loader, desc='Collecting logits'):
                imgs = imgs.to(device)
                out = model(imgs)
                all_logits.append(out['binary_logit'].cpu())
                all_labels.append(labels)
        all_logits = torch.cat(all_logits).to(device)
        all_labels = torch.cat(all_labels).to(device)

        def eval_nll():
            optimizer.zero_grad()
            loss = F.cross_entropy(self(all_logits), all_labels)
            loss.backward()
            return loss

        optimizer.step(eval_nll)
        print(f'Optimal temperature: T = {self.temperature.item():.4f}')
        return self


temp_scaler = TemperatureScaler()
temp_scaler.calibrate(model, val_loader, DEVICE)
T_opt = temp_scaler.temperature.item()
print(f'\nTemperature T={T_opt:.4f}')
print(f'T > 1: model was overconfident -> now more calibrated' if T_opt > 1 else
      f'T < 1: model was underconfident -> now more calibrated')

In [ ]:
# Cell 9 — Calibration quality: reliability diagram

def get_calibration_data(model, loader, device, temperature=1.0):
    """Collect probabilities and labels for calibration analysis."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='Collecting'):
            imgs = imgs.to(device)
            out = model(imgs)
            logits = out['binary_logit'] / temperature
            probs = torch.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_probs), np.array(all_labels)


def reliability_diagram(probs, labels, n_bins=10, title=''):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_sizes = [], [], []
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i+1])
        if mask.sum() == 0:
            continue
        bin_accs.append(labels[mask].mean())
        bin_confs.append(probs[mask].mean())
        bin_sizes.append(mask.sum())

    ece = sum(abs(a - c) * s for a, c, s in zip(bin_accs, bin_confs, bin_sizes)) / sum(bin_sizes)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax.bar(bin_confs, bin_accs, width=0.08, alpha=0.6, color='steelblue', label='Model')
    ax.set_xlabel('Confidence (predicted probability)')
    ax.set_ylabel('Accuracy (fraction fake)')
    ax.set_title(f'{title}  ECE={ece:.4f}')
    ax.legend()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    return fig, ece


# Before calibration
probs_before, labels_arr = get_calibration_data(model, test_loader, DEVICE, temperature=1.0)
fig_before, ece_before = reliability_diagram(probs_before, labels_arr, title='Before Calibration')
plt.savefig(f'{CKPT_DIR}/calibration_before.png', dpi=150)
plt.show()

# After calibration
probs_after, _ = get_calibration_data(model, test_loader, DEVICE, temperature=T_opt)
fig_after, ece_after = reliability_diagram(probs_after, labels_arr, title='After Temperature Scaling')
plt.savefig(f'{CKPT_DIR}/calibration_after.png', dpi=150)
plt.show()

print(f'ECE before: {ece_before:.4f}')
print(f'ECE after:  {ece_after:.4f}  (lower is better)')
print(f'Improvement: {(ece_before - ece_after) / ece_before * 100:.1f}%')

In [ ]:
# Cell 10 — Save final hardened + calibrated model

FINAL_HARDENED = f'{CKPT_DIR}/deeptruth_image_hardened_final.pth'

torch.save({
    'model_state': model.state_dict(),
    'temperature': T_opt,
    'fake_type_map': FAKE_TYPE_MAP,
    'ece_before': ece_before,
    'ece_after': ece_after,
    'adv_history': history_adv,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std': IMAGENET_STD,
}, FINAL_HARDENED)

print(f'Saved hardened model: {FINAL_HARDENED}')
print(f'Temperature T = {T_opt:.4f}')
print(f'ECE: {ece_before:.4f} -> {ece_after:.4f}')
print('\nReady for notebook 06 (ONNX export).')